In [1]:
import datasets

processed_testset_path = '/data/text-mol/data/Mol-LLM-v7.1/mol_llm_testset_general'
testset = datasets.load_from_disk(processed_testset_path)
testset = testset.map(
    lambda example, idx: {"idx": idx},
    with_indices=True,
)

/home/chanhui-lee/miniconda3/envs/mol-llama/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import rdkit
from rdkit import Chem
from rdkit.Chem import AllChem
import random

smiles = testset[0]['smiles']
from rdkit import Chem
from rdkit.Chem import AllChem
import random

def get_3d_coordinates(smiles):
    """Generate approximate 3D coordinates for a molecule from SMILES.
       Always returns at least some coordinates, even if embedding fails."""
    
    # Try creating RDKit molecule
    mol = Chem.MolFromSmiles(smiles)
    if mol is None or mol.GetNumAtoms() == 0:
        # If SMILES totally invalid, return random dummy atom of the same sized carbon chain
        return {
            'cid': '0000000',
            'atoms': ['C'] * mol.GetNumAtoms(),
            'coordinates': [[random.uniform(-2, 2), random.uniform(-2, 2), random.uniform(-2, 2)]
                           for _ in range(mol.GetNumAtoms())]
        }

    try:
        mol = Chem.AddHs(mol)
    except Exception:
        # fallback: no AddHs
        pass
    
    atoms = [atom.GetSymbol() for atom in mol.GetAtoms() if atom.GetSymbol() != 'H']
    
    # Default coordinate fallback in case we never embed properly
    coordinates = [[random.uniform(-2, 2), random.uniform(-2, 2), random.uniform(-2, 2)]
                   for _ in atoms]
    
    # Attempt normal embedding
    try:
        params = AllChem.ETKDGv3()
        result = AllChem.EmbedMolecule(mol, params)
        
        # Retry with random coords if ETKDG fails
        if result != 0:
            result = AllChem.EmbedMolecule(mol, useRandomCoords=True)
        
        # If embedding successful, get coordinates
        if mol.GetNumConformers() > 0:
            AllChem.UFFOptimizeMolecule(mol, maxIters=100)
            conf = mol.GetConformer()
            coordinates = []
            for atom in mol.GetAtoms():
                if atom.GetSymbol() != 'H':
                    pos = conf.GetAtomPosition(atom.GetIdx())
                    coordinates.append([pos.x, pos.y, pos.z])
            
    except Exception as e:
        # Any RDKit embedding exception — fallback to random coords
        pass
    
    dict_data = {
        'atoms': atoms,
        'coordinates': [coordinates]
    }
    return dict_data

def process_3d_coordinates(instance):
    smiles = instance['smiles']
    dict_data = get_3d_coordinates(smiles)
    cid = instance['idx']
    dict_data.update({'cid': cid})
    instance['3d_info'] = dict_data
    return instance

In [3]:
info_3d_testset = testset.map(process_3d_coordinates, num_proc=200)

Map (num_proc=200):   0%|          | 0/55757 [00:00<?, ? examples/s][04:10:26] UFFTYPER: Warning: hybridization set to SP3 for atom 3
[04:10:26] UFFTYPER: Warning: hybridization set to SP3 for atom 18
[04:10:26] UFFTYPER: Unrecognized hybridization for atom: 1
Map (num_proc=200):   0%|          | 4/55757 [00:02<7:45:47,  1.99 examples/s][04:10:26] UFFTYPER: Unrecognized atom type: Co+3 (1)
[04:10:26] UFFTYPER: Warning: hybridization set to SP3 for atom 4
[04:10:26] UFFTYPER: Warning: hybridization set to SP3 for atom 7
[04:10:26] UFFTYPER: Warning: hybridization set to SP3 for atom 16
[04:10:26] UFFTYPER: Unrecognized hybridization for atom: 29
[04:10:26] UFFTYPER: Unrecognized atom type: Pt+2 (29)
Map (num_proc=200):   0%|          | 29/55757 [00:02<50:28, 18.40 examples/s] [04:10:26] UFFTYPER: Warning: hybridization set to SP3 for atom 30
[04:10:26] UFFTYPER: Unrecognized hybridization for atom: 1
[04:10:26] UFFTYPER: Unrecognized atom type: Co+3 (1)
[04:10:26] UFFTYPER: Unrecognized

In [4]:
# save the testset
info_3d_testset_path = processed_testset_path + '_3d_info-v2'
info_3d_testset.save_to_disk(info_3d_testset_path)

Saving the dataset (0/2 shards):   0%|          | 0/55757 [00:00<?, ? examples/s]

Saving the dataset (2/2 shards): 100%|██████████| 55757/55757 [00:01<00:00, 27891.69 examples/s]


In [5]:
info_3d = info_3d_testset['3d_info'][:]
testset_stmiles = testset['smiles'][:]
# update info_3d with smiles
for i, info in enumerate(info_3d):
    info['smiles'] = testset_stmiles[i]


In [6]:
save_dir = '/home/chanhui-lee/Mol-LLaMA/data/mol_llm_testset'

In [ ]:
import os
import json


# save info_3d
with open(os.path.join(save_dir, 'test_mol.json'), 'w') as f:
    json.dump(info_3d, f, indent=4)
with open(os.path.join(save_dir, 'train_mol.json'), 'w') as f:
    json.dump(info_3d, f, indent=4)


In [15]:
# save debugging info_3d
with open(os.path.join(save_dir + '_debugging', 'test_mol.json'), 'w') as f:
    json.dump(info_3d[:100], f, indent=4)
with open(os.path.join(save_dir + '_debugging', 'train_mol.json'), 'w') as f:
    json.dump(info_3d[:100], f, indent=4)

In [9]:
testset[0].keys()

dict_keys(['task', 'x', 'edge_index', 'edge_attr', 'additional_x', 'additional_edge_index', 'additional_edge_attr', 'input_mol_string', 'prompt_text', 'target_text', 'smiles', 'instruction', 'target', 'idx'])

In [10]:
def process_instance(instance):
    question = instance['instruction']
    question = question.replace('<INPUT>', 'the molecule')
    target = instance['target']

    system = "You are a helpful assistant specializing in chemistry and biology. The instruction that describes a task is given, paired with molecules. Provide a response that appropriately completes the request.\n\nNotice that here are some rules you need to follow:\n1. Please give me your ANSWER for the given instances in the format 'Answer: ...'"
    conversations = [
        {
            "user": "Molecule: <mol>\nQuestion: {question}\n".format(question=question),
            "assistant": target
        }
    ]

    instance['system'] = system
    instance['conversations'] = conversations
    instance['category'] = instance['task']
    instance['qid'] = instance['idx']
    instance['cid'] = instance['idx']
    return instance



In [11]:
text_testset = testset.map(process_instance, num_proc=200)
text_testset.save_to_disk(os.path.join(save_dir, 'test'))
text_testset.save_to_disk(os.path.join(save_dir, 'train'))


Saving the dataset (1/1 shards): 100%|██████████| 55757/55757 [00:01<00:00, 30257.69 examples/s]


In [16]:
import random
random_indices = random.sample(range(len(text_testset)), 100)
debugging_testset = text_testset.select(random_indices)
debugging_testset.save_to_disk(os.path.join(save_dir + '_debugging', 'test'))
debugging_testset.save_to_disk(os.path.join(save_dir + '_debugging', 'train'))

Saving the dataset (1/1 shards): 100%|██████████| 100/100 [00:00<00:00, 1658.40 examples/s]


In [18]:
debugging_testset[0]

{'task': 'orderly-forward_reaction_prediction',
 'x': [[5, 0, 4, 5, 3, 0, 2, 0, 0],
  [5, 0, 4, 5, 0, 0, 2, 0, 0],
  [5, 0, 4, 5, 3, 0, 2, 0, 0],
  [5, 0, 4, 5, 3, 0, 2, 0, 0],
  [7, 0, 2, 5, 0, 0, 1, 0, 0],
  [5, 0, 3, 5, 0, 0, 1, 0, 0],
  [7, 0, 1, 5, 0, 0, 1, 0, 0],
  [6, 0, 3, 5, 0, 0, 1, 0, 1],
  [5, 0, 4, 5, 2, 0, 2, 0, 1],
  [5, 0, 4, 5, 2, 0, 2, 0, 1],
  [5, 2, 4, 5, 0, 0, 2, 0, 1],
  [6, 0, 3, 5, 2, 0, 2, 0, 0],
  [5, 0, 4, 5, 2, 0, 2, 0, 0],
  [7, 0, 2, 5, 1, 0, 2, 0, 0],
  [5, 0, 4, 5, 2, 0, 2, 0, 1],
  [8, 0, 1, 5, 0, 0, 2, 0, 0],
  [5, 0, 3, 5, 0, 0, 1, 1, 1],
  [5, 0, 3, 5, 0, 0, 1, 1, 1],
  [16, 0, 1, 5, 0, 0, 2, 0, 0],
  [6, 0, 2, 5, 0, 0, 1, 1, 1],
  [5, 0, 3, 5, 0, 0, 1, 1, 1],
  [6, 0, 3, 5, 0, 0, 1, 0, 1],
  [5, 0, 4, 5, 2, 0, 2, 0, 1],
  [5, 0, 4, 5, 2, 0, 2, 0, 1],
  [7, 0, 2, 5, 0, 0, 2, 0, 1],
  [5, 0, 4, 5, 2, 0, 2, 0, 1],
  [5, 0, 4, 5, 2, 0, 2, 0, 1],
  [6, 0, 2, 5, 0, 0, 1, 1, 1],
  [5, 0, 3, 5, 0, 0, 1, 1, 1],
  [16, 0, 1, 5, 0, 0, 2, 0, 0]],
 'edge_index':